Using POS-Tagging, Dependency-Parsing on Samsung reviews to find the qualitative aspects of top 5 / 10 features

In [7]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

In [3]:
path = r'C:\Users\arnig\Documents\Coding_2024\python_work\UpGrad\DS_C70\Specialization_NLP\Syntactic-Processing\Syntactic-Processing_upgrad\POS Tagging Case Study\Dataset\Samsung.txt'
with open(path, mode='r', encoding='utf-8') as f_open:
    reviews_data = f_open.read()

print(type(reviews_data))
print(len(reviews_data))

<class 'str'>
7488235


In [4]:
# Splitting into sentences
reviews = reviews_data.split('\n')
reviews[:10]

["I feel so LUCKY to have found this used (phone to us & not used hard at all), phone on line from someone who upgraded and sold this one. My Son liked his old one that finally fell apart after 2.5+ years and didn't want an upgrade!! Thank you Seller, we really appreciate it & your honesty re: said used phone.I recommend this seller very highly & would but from them again!!",
 'nice phone, nice up grade from my pantach revue. Very clean set up and easy set up. never had an android phone but they are fantastic to say the least. perfect size for surfing and social media. great phone samsung',
 'Very pleased',
 'It works good but it goes slow sometimes but its a very good phone I love it',
 'Great phone to replace my lost phone. The only thing is the volume up button does not work, but I can still go into settings to adjust. Other than that, it does the job until I am eligible to upgrade my phone again.Thaanks!',
 'I originally was using the Samsung S2 Galaxy for Sprint and wanted to retu

In [ ]:
# Top features
nouns = []
for doc in tqdm(reviews):
    tokens = nlp(doc)
    nouns.extend(token.text.lower() for token in tokens if token.pos_ == 'NOUN')

nouns = pd.Series(nouns).value_counts(normalize= True)

100%|██████████| 46355/46355 [02:12<00:00, 349.78it/s]


In [46]:
# Top 10 features from reviews
features = nouns.head(5).index.values
features

array(['phone', 'battery', 'product', 'screen', 'time'], dtype=object)

In [ ]:
# Filtering reviews based on presence of top feature words
feature_reviews = {feature: [doc for doc in reviews if feature in doc] for feature in features}

Filtering qualities based on pos and dependency of qualifiers with feature words.

In [59]:
feature_qualities = {}
for feature, docs in feature_reviews.items():
    qualities = []
    for doc in tqdm(docs):
        nearby = lambda i: (doc[i-2], doc[i-1], doc[i], doc[i+1], doc[i+2])
        qualities.extend([tok.text.lower() for tok in nlp(doc) 
                          if (tok.pos_ == 'ADJ' or tok.pos_ == 'ADV') and feature in list(tok.ancestors)])
    feature_qualities[feature] = pd.Series(qualities)

100%|██████████| 3675/3675 [00:29<00:00, 122.98it/s]


In [60]:
for key, val in feature_qualities.items():
    feature_qualities[key] = ' '.join(val.index.values)

feature_qualities

{'phone': '', 'battery': '', 'product': '', 'screen': '', 'time': ''}

Due to syntactical constraints dependency parsing does not always reveal the context of usage. But if dependency condition is removed then the pos-tags reveal usage of qualifiers without the context of usage of such qualifiers.

In [61]:
feature_qualities = {}
for feature, docs in feature_reviews.items():
    qualities = []
    for doc in tqdm(docs):
        nearby = lambda i: (doc[i-2], doc[i-1], doc[i], doc[i+1], doc[i+2])
        qualities.extend(set([tok.text.lower() for tok in nlp(doc) if tok.pos_ == 'ADJ']))
    feature_qualities[feature] = pd.Series(qualities).value_counts(normalize= True).head(5)

100%|██████████| 3675/3675 [00:29<00:00, 122.71it/s]


In [62]:
feature_qualities

{'phone': great    0.074688
 good     0.049170
 new      0.030192
 nice     0.020460
 best     0.016863
 Name: proportion, dtype: float64,
 'battery': great    0.040694
 good     0.035659
 new      0.022057
 other    0.016871
 only     0.014955
 Name: proportion, dtype: float64,
 'product': good         0.083521
 great        0.082480
 excellent    0.071367
 new          0.026828
 happy        0.020837
 Name: proportion, dtype: float64,
 'screen': great    0.036891
 good     0.029887
 new      0.018723
 big      0.016885
 other    0.016746
 Name: proportion, dtype: float64,
 'time': great    0.040822
 good     0.033638
 new      0.021896
 other    0.017234
 more     0.014229
 Name: proportion, dtype: float64}

Time is probably delivery time